# D74 收尾①③ —— 冲突归因分析 + final-test

**①** 冲突归因(val): 把 −32% 拆成「赢在哪」——按拥挤度分箱的延误差 + 放行顺序偏离率 + Top-10 案例。
**③** final-test(D43): test 集全程封存, **此格只跑一次**; 跑前自动备份 json。

## 设置

In [ ]:
import sys, os
venv_site = '/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/lib/python3.11/site-packages'
if os.path.isdir(venv_site) and venv_site not in sys.path:
    sys.path.insert(0, venv_site)
%cd /rds/homes/h/hxn886/ondemand/final_project
PROJ='/rds/homes/h/hxn886/ondemand/final_project'
PY='/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/bin/python'
PREFIX=(f'cd {PROJ} && env PYTHONUNBUFFERED=1 PYTHONPATH=. '
        f'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True MPLBACKEND=Agg {PY}')
print('OK')


## ① 冲突归因分析(val, 约几分钟)

In [ ]:
!{PREFIX} scripts/37_conflict_analysis.py --ckpt artifacts/reports/ppo_d73_AB_s0_best.pt --split val --with-signaller


## ③ final-test(**只跑一次**; resume 已完成→零训练, 直接用 best 在 test 全日评估并写回 json)

In [ ]:
!cp artifacts/reports/ppo_d73_AB_s0.json artifacts/reports/ppo_d73_AB_s0_preTest.json   # 备份, 不覆盖
!{PREFIX} scripts/13_train_ppo.py \
    --run-tag ppo_d73_AB_s0 --feature-version 2 --seed 0 \
    --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt \
    --bc-anchor-beta 0.01 --lr 5e-5 \
    --encoder gnn --control-classes 12 \
    --kl-anchor schedule --kl-beta 0.03 --reserve-window wide \
    --curriculum --curriculum-tiers 185,286,300,999999 \
    --promote-at 1.0 --promote-delay-k 1.0 \
    --train-eval-days 6 --iters 100 --steps-per-iter 8192 --eval-every 5 \
    --early-stop-patience 8 --best-rescue --resume --final-test


## ①-test 口径的规则对照(test 上的 fcfs/信号员参照, 供 headline 表)

In [ ]:
!{PREFIX} scripts/37_conflict_analysis.py --ckpt artifacts/reports/ppo_d73_AB_s0_best.pt --split test --with-signaller


## 汇总打印

In [ ]:
import json
d=json.load(open('artifacts/reports/ppo_d73_AB_s0.json'))
ft=d.get('final_test')
print('final_test:', {k:round(v,1) for k,v in ft.items() if isinstance(v,(int,float))} if ft else '(未跑)')
ca=json.load(open('artifacts/reports/conflict_analysis_val.json'))
print('\nval 拥挤度分箱 Δ(RL−fcfs):')
for r in ca['bins']: print(f"  {r['bin']:<8} n={r['n']:>5}  Δ {r['d']:>12.0f}")
print(f"顺序偏离率: RL {ca['inv_rl']:.3f} vs fcfs {ca['inv_fcfs']:.3f}")


## test 全基线表(补 headline 的规则参照; 只读评估, 可重复跑)

In [ ]:
# test 全基线表(headline 配套; 规则策略, 无 torch 推理, ~几分钟)
import json
from derbysim.learn.features import Featurizer
from derbysim.learn.panel import PanelAdapter
from derbysim.learn.baselines import (lookup_table, make_lookup_policy, make_ars_proxy_policy,
                                      make_fcfs_policy, make_fsfs_policy)
from derbysim.learn.rollout import (as_factory, make_greedy_set_policy,
                                    signaller_replay_factory, evaluate_policy)
from derbysim.learn.scenarios import build_day_trains, day_horizon, load_assets, operating_days
from derbysim.sim.reward import RewardedDynamicEnv
net,params,dwell,ptc=load_assets(); ad=PanelAdapter(net,Featurizer(net,ptc))
envf=lambda: RewardedDynamicEnv(net,params,ptc,dwell,proactive=True,reserve_window='wide',control_classes={'1','2'})
eps=[(str(w['start'])[:16],(lambda w=w: build_day_trains(w)),day_horizon(w)) for w in operating_days('test')]
top=lookup_table()
out={}
for tag,fac in [('signaller',signaller_replay_factory),('fcfs',make_fcfs_policy(top)),
                ('lookup',make_lookup_policy(top)),('fsfs',make_fsfs_policy(top)),
                ('always_set',as_factory(make_greedy_set_policy()))]:
    b=evaluate_policy(envf,ad,fac,eps); b.pop('eval_episodes',None); out[tag]=b
    print(f"{tag:<11} 完成 {b['eval_completion_rate']:.4f} | 延误 {b['eval_weighted_secondary_delay_s']:>11.0f} | reward {b['eval_episode_reward']:>9.1f}")
json.dump(out,open('artifacts/reports/test_baselines.json','w'))
print('saved -> artifacts/reports/test_baselines.json  (与 ppo_d73_AB_s0_test.json 的 final_test 同口径对照)')
